- Read QuakeML
- Filter catalog
- Define template parameters
- Download miniseeds surrounding the templates
- Create template objects from miniseeds, save
- Perform detection using template matching

In [19]:
import obspy
import eqcorrscan
import numpy as np
import pandas as pd

### Read in starting earthquake catalog, in QuakeML format

In [2]:
cat = obspy.core.event.catalog.read_events('endeavour_june2017.xml')

### Filter earthquake catalog as desired

In [ ]:
min_lat = 47.9
max_lat = 48.05
min_lon = -129.15
max_lon = -129.05
min_time = 
max_time = 
min_magnitude = 1.2

In [3]:
cat = cat.filter("time > 2017-06-01","time < 2017-06-30","longitude > -129.15","longitude < -129.05","latitude < 48.05","latitude > 47.9","magnitude > 1.2")

### Download waveform data for the catalog duration

In [6]:
# Get list of stations and channels from QuakeML
picks = [p.picks for p in cat.events]
picks = sum(picks,[])

networks = [p.waveform_id.network_code for p in picks]
stations = [p.waveform_id.station_code for p in picks]
channels = [p.waveform_id.channel_code[0:2] + '*' for p in picks]
# Toss pressure channels:
channelToRemove = 'HD*'
channels = [value for value in channels if value != channelToRemove]

sta_list = [f"{n}.{s}..{c[0:2]}" for n, s, c in zip(networks,stations,channels)]
sta_list = np.unique(sta_list)

# Get all the info for those stations from IRIS
network = ",".join((np.unique(networks)).tolist())
channel = ",".join((np.unique(channels)).tolist())
station = ",".join((np.unique(stations)).tolist())


## Miniseed naming conventions:
KEMF.NV.2017.yearday

In [14]:
# Loop over days and download each (unprocessed) into a directory
path = 'data_june2017/'
t1 = obspy.UTCDateTime(2017,6,1)
t2 = obspy.UTCDateTime(2017,6,3)

client = obspy.clients.fdsn.client.Client('IRIS')

time_bins = np.arange(t1.datetime,t2.datetime,pd.Timedelta(1,'days'))
time_bins = [pd.to_datetime(t) for t in time_bins]

for t in time_bins:
    
    t1 = obspy.UTCDateTime(t)
    t2 = obspy.UTCDateTime(t + pd.Timedelta(1,'day'))
    
    pathname = path + t1.strftime("%Y%m%d")+'.mseed'
    
    st = client.get_waveforms(network,station,'',channel,t1,t2)
    
    st.write(pathname)

### Define template parameters

In [15]:
filter_lowpass = 8.0
filter_highpass = 35.0
samp_rate = 200.0
length = 0.5
prepick = 0.05
process_len = 3600
min_snr = 0.1

### Create templates from miniseeds

In [ ]:
# Start with only a few events
filt_cat = cat[0:2]

template_list = []
for i,ev in enumerate(filt_cat):

    # HERE ADD IN NAME OF STREAM BY DATE OF EVENT
    filename = path
    st = obspy.read(filename)
    temp_cat = obspy.core.event.Catalog(events=[ev])

    # Retrieve saved waveform and process it
    template_st = eqcorrscan.core.template_gen.template_gen(method=“from_meta_file”, st=st,lowcut=8.0, highcut=35.0, samp_rate=200.0, length=0.5,
    filt_order=4, prepick=0.05,meta_file=temp_cat, data_pad=20.,
    process_len=3600, min_snr=2.0, parallel=True,swin=‘all’,delayed=True)

    # Make template from processed waveform
    template = eqcorrscan.core.match_filter.template.Template(name=ev.resource_id.id, st=template_st[0], lowcut=8.0, highcut=35.0, samp_rate=200.0, filt_order=4, process_length=3600, prepick=0.05, event=ev)
    template_list.append(template)
    
tribe = eqcorrscan.core.match_filter.tribe.Tribe(templates=template_list)
for i,t in enumerate(tribe):
    tribe[i].event.origins[0].resource_id = t.name

In [ ]:
# Write templates to file, if desired
tribe.write('filename')

### Perform detection on miniseed data

In [ ]:
# Define detection parameters
threshold = 
threshold_type = 
trig_int = 
# etc.

In [ ]:
t1 = obspy.UTCDateTime(2017,6,1)
t2 = obspy.UTCDateTime(2017,6,3)
time_bins = np.arange(t1.datetime,t2.datetime,pd.Timedelta(1,‘days’))
time_bins = [pd.to_datetime(t) for t in time_bins]

for t in time_bins:
    t1 = obspy.UTCDateTime(t)
    t2 = obspy.UTCDateTime(t + pd.Timedelta(1,‘day’))
    pathname = path + t1.strftime("%Y%m%d")+'.mseed'
    print(pathname)
    stream = obspy.core.stream.read(pathname)
    test = tribe.detect(stream, threshold=0.4, threshold_type=‘av_chan_corr’,trig_int=1,parallel_process=False)